环境配置

In [ ]:
import math
import sys
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(10)

WEEK19 = Path.cwd()          
assert (WEEK19 / "0921" / "raw" / "cmn.txt").exists(), f"路径不对: {WEEK19}"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class Cfg:
    # ── 模型 ──
    d_model = 128             
    n_head = 4                # 先 4 头
    d_k = d_model // n_head   # 32
    d_v = d_model // n_head   # 32
    d_ff = 256                # 论文是 4 * d_model（512），先减半换速度
    n_layers = 2              # 论文 6 层
    dropout = 0.1
    max_len = 100             # 与数据预处理的最大长度一致（位置编码表按它建）
    # ── 特殊符
    pad_idx, sos_idx, eos_idx, unk_idx = 0, 1, 2, 3
    # ── 训练 ──
    batch_size = 64
    lr = 1e-3


print("device:", DEVICE)
print("d_k = d_v =", Cfg.d_k)

两种mask

In [ ]:
def pad_mask(seq, pad_idx=Cfg.pad_idx):
    """(b, l) 词 id 序列 → (b, 1, l)；True = 真实词（可见）。"""
    return (seq != pad_idx).unsqueeze(-2) 

def causal_mask(l, device = None):
    """长度 l → (1, l, l)；下三角为 True（位置 i 只能看自己及之前）。"""
    return torch.ones(1, l, l,
                      dtype=torch.bool,
                      device=device,).tril()


正弦位置编码

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=Cfg.max_len):
        super().__init__()
        assert d_model % 2 == 0
        position = torch.arange(
            max_len,
            dtype=torch.float32,
        ).unsqueeze(1)
        
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32,)*
            (-math.log(10000.0)/d_model)
        )
        
        table = torch.zeros(max_len,d_model)
        table[:,0::2] = torch.sin(position*div_term)
        table[:,1::2] = torch.cos(position*div_term)
        
        self.register_buffer("table", table)
        
    def forward(self, x):
        """x: (b, l, d_model) → 同形状。"""
        seq_len = x.size(1)
        
        if seq_len > self.table.size(0):
            raise ValueError(
                f"序列长度 {seq_len} 超过位置编码上限 "
                f"{self.table.size(0)}"
            )
        return x + self.table[:seq_len].unsqueeze(0)



ScaledDotProductAttention

In [ ]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        """
        q: (b, h, lq, d_k)
        k: (b, h, lk, d_k)
        v: (b, h, lk, d_v)
        mask: (b, 1, lq, lk)，True = 可见；None 表示不遮
        返回 (context (b, h, lq, d_v), attn (b, h, lq, lk))
        """
        attn = torch.matmul(q, k.transpose(2,3))
        d_k = q.size(-1)
        attn = attn / math.sqrt(d_k)
        
        if mask is not None:
            mask = mask.to(dtype=torch.bool)
            attn = attn.masked_fill(~mask, float("-inf"))
            
        weights = F.softmax(attn, dim=-1)
        dropped_weights = self.dropout(weights)
        context = torch.matmul(dropped_weights, v)
        
        return context, weights



MultiHeadAttention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head, d_k, d_v, dropout=0.1):
        super().__init__()
        
        self.n_head = n_head
        self.d_k = d_k
        self.d_v = d_v
        
        self.w_q = nn.Linear(d_model, n_head* d_k, bias=False)
        self.w_k = nn.Linear(d_model, n_head* d_k, bias=False)
        self.w_v = nn.Linear(d_model, n_head* d_v, bias=False)
        
        self.fc = nn.Linear(n_head* d_v, d_model, bias=False)
        
        self.attention = ScaledDotProductAttention(dropout=dropout)

    def forward(self, q, k, v, mask=None):
        """
        q: (b, lq, d_model)、k: (b, lk, d_model)、v: (b, lk, d_model)
        mask: 按 M3 的语义传；补维（unsqueeze）在本函数里做
        返回 (b, lq, d_model)，顺手把 attn 权重也返回更好（画图用）
        """
        batch_size = q.size(0)
        len_q = q.size(1)
        len_k = k.size(1)
        len_v = v.size(1)
        
        if len_k != len_v:
            raise ValueError(
                f"K和V的序列长度必须相同，"
                f"现在是len_k={len_k}, len_v={len_v}"
            )
            
        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)
        
        q = q.view(batch_size, len_q, self.n_head, self.d_k).transpose(1,2)
        k = k.view(batch_size, len_k, self.n_head, self.d_k).transpose(1,2) 
        v = v.view(batch_size, len_v, self.n_head, self.d_v).transpose(1,2)
        
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)
            elif mask.dim() != 4:
                raise ValueError(
                    "mask应为3维或4维,"
                    f"实际形状是{tuple(mask.shape)}"
                )
                
        context, weights = self.attention(q, k, v, mask=mask)
        context = (context.transpose(1,2)
                   .contiguous()
                   .view(batch_size, len_q, self.n_head* self.d_v))
        
        output = self.fc(context)
        
        return output, weights


位置编码+两个子层

In [ ]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        """(b, l, d_model) → 同形状。"""
        hidden = self.w1(x)
        hidden = F.relu(hidden)
        hidden = self.dropout(hidden)
        output = self.w2(hidden)
        
        return output


class EncoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_head, d_k, d_v, dropout):
        super().__init__()
        
        self.self_attn = MultiHeadAttention(d_model, n_head, d_k, d_v, dropout,)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout,)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        

    def forward(self, x, src_mask):
        """x: (b, l, d_model) → 同形状；src_mask: (b, 1, l)。"""
        attn_out, _ = self.self_attn(x, x, x, mask=src_mask,)
        x = self.norm1(x + self.dropout1(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        
        return x 


class DecoderLayer(nn.Module):
    def __init__(self, d_model, d_ff, n_head, d_k, d_v, dropout):
        super().__init__()
        
        self.self_attn = MultiHeadAttention(d_model, n_head, d_k, d_v, dropout,)
        self.cross_attn = MultiHeadAttention(d_model, n_head, d_k, d_v, dropout,)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout,)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, trg_mask, src_mask, return_attn=False):
        """x: (b, lt, d_model)、enc_out: (b, ls, d_model) → 同 x 形状。"""
        self_out, self_weights = self.self_attn(x, x, x, mask=trg_mask,)
        x = self.norm1(x + self.dropout1(self_out))
        
        cross_out, cross_weights = self.cross_attn(x, enc_out, enc_out, mask=src_mask,)
        x = self.norm2(x + self.dropout2(cross_out))
        
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_out))

        if return_attn:
            return x, self_weights, cross_weights

        return x



组装encoder和decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_head, d_k, d_v, d_ff, dropout, max_len):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=Cfg.pad_idx,)
        self.position = PositionalEncoding(d_model, max_len,)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderLayer(d_model,d_ff,n_head,d_k, d_v, dropout,)
            for _ in range(n_layers)
        ])

    def forward(self, src, src_mask):
        """src: (b, ls) 词 id → (b, ls, d_model)。"""
        x = self.embedding(src)
        x = x * math.sqrt(self.d_model)

        x = self.position(x)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x, src_mask)
        return x


class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_head, d_k, d_v, d_ff, dropout, max_len):
        super().__init__()
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=Cfg.pad_idx,)

        self.position = PositionalEncoding(d_model, max_len,)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model,d_ff, n_head, d_k, d_v, dropout,)
            for _ in range(n_layers)
        ])

    def forward(self, trg, enc_out, trg_mask, src_mask, return_attn=False):
        """trg: (b, lt) 词 id → (b, lt, d_model)。"""
        x = self.embedding(trg)
        x = x * math.sqrt(self.d_model)

        x = self.position(x)
        x = self.dropout(x)

        last_self_weights = None
        last_cross_weights = None

        for layer_index, layer in enumerate(self.layers):
            is_last_layer = (
                layer_index == len(self.layers) - 1
            )

            if return_attn and is_last_layer:
                (x, last_self_weights, last_cross_weights,) = layer(
                    x, enc_out, trg_mask, src_mask, return_attn=True,)
            else:
                x = layer(x, enc_out, trg_mask, src_mask,)

        if return_attn:
            return (x, last_self_weights, last_cross_weights,)

        return x


class Transformer(nn.Module):
    def __init__(self, src_vocab_size, trg_vocab_size, max_len):
        super().__init__()
        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            d_model=Cfg.d_model,
            n_layers=Cfg.n_layers,
            n_head=Cfg.n_head,
            d_k=Cfg.d_k,
            d_v=Cfg.d_v,
            d_ff=Cfg.d_ff,
            dropout=Cfg.dropout,
            max_len=max_len,
        )

        self.decoder = Decoder(
            vocab_size=trg_vocab_size,
            d_model=Cfg.d_model,
            n_layers=Cfg.n_layers,
            n_head=Cfg.n_head,
            d_k=Cfg.d_k,
            d_v=Cfg.d_v,
            d_ff=Cfg.d_ff,
            dropout=Cfg.dropout,
            max_len=max_len,
        )

        self.output_projection = nn.Linear(
            Cfg.d_model,
            trg_vocab_size,
        )

    def forward(self, src, trg, return_attn=False):
        """返回 (b, l, trg_vocab) 的 logits。"""
        src_mask = pad_mask(src,Cfg.pad_idx,)
        trg_pad_mask = pad_mask(trg,Cfg.pad_idx,)
        trg_causal_mask = causal_mask(trg.size(1),device=trg.device,)
        trg_mask = (trg_pad_mask & trg_causal_mask)

        enc_out = self.encoder(src, src_mask,)

        if return_attn:
            (dec_out,self_weights,cross_weights,
            ) = self.decoder(trg,enc_out,trg_mask,src_mask,return_attn=True,)
        else:
            dec_out = self.decoder(
                trg,enc_out,trg_mask,src_mask,
            )
    
        logits = self.output_projection(dec_out)

        if return_attn:
            return (logits, self_weights, cross_weights,)
        return logits
